<a href="https://colab.research.google.com/github/advayc/tencent-auk/blob/main/tencent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tencent AuK on a free Colab T4 — local experimental GGUF run

This notebook runs Tencent AuK Base weights locally through the experimental audio.cpp GGUF runtime. It does not call the shared public Gradio Space, so it avoids the 502/rate-limit issue, and it does not use the earlier TTS-splice, denoiser, or VAD fallback.

It is deliberately pinned to a free Tesla T4 (16 GB). The quality-first profile keeps the AuK Base generator and VAE in F32, uses a Q8_0 Qwen conditioner, and releases stages between requests. This is a genuine AuK-weight execution route, but it is experimental: listen to every result before presenting it as a finished edit.

The ready-made cells cover transcript editing, tone/emotion editing, speech cleanup, and speaker isolation. Use only recordings you own or are authorized to edit.

## What to do

1. In Colab select Runtime → Change runtime type → T4 GPU, then reconnect.
2. Run the setup cells in order through Configure the T4 profile. The one-time model download is about 13 GB; the audio runtime itself is downloaded as a prebuilt T4 server, so no CUDA source build is required.
3. Run Examples 1–3 one at a time. They are the three labelled before/after pairs for the demo. No upload is needed: setup downloads Tencent's sample audio.
4. Run Example 4 for speaker isolation and listen to all inline players.
5. After listening, mark only the pairs you approve and download the ZIP.

Keep clips around 4–8 seconds and run one request at a time. If the F32 Base profile reports a CUDA out-of-memory error, the notebook prints a warning and retries once with its explicitly labelled Base-Q8 lower-memory fallback.

In [ ]:
# T4-only preflight. Do not import torch or torchaudio in this notebook.
import re
import shutil
import subprocess

try:
    gpu_line = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.used', '--format=csv,noheader'],
        text=True,
    ).strip().splitlines()[0]
except (FileNotFoundError, subprocess.CalledProcessError) as exc:
    raise RuntimeError('No NVIDIA GPU is attached. In Colab choose Runtime → Change runtime type → T4 GPU, then reconnect.') from exc

gpu_name, gpu_total, gpu_used = [part.strip() for part in gpu_line.split(',')]
gpu_total_mib = float(re.search(r'[0-9.]+', gpu_total).group())
free_disk_gib = shutil.disk_usage('/content').free / 2**30
print(f'GPU: {gpu_name} | total: {gpu_total} | used: {gpu_used}')
print(f'Free /content storage: {free_disk_gib:.1f} GiB')
if 'T4' not in gpu_name.upper():
    raise RuntimeError(f'This notebook is intentionally configured for a Tesla T4, but Colab attached {gpu_name}. Select T4 GPU and reconnect.')
if gpu_total_mib < 14000:
    raise RuntimeError(f'The detected T4 has unexpectedly little VRAM: {gpu_total}. Reconnect and try again.')
if free_disk_gib < 18:
    raise RuntimeError('At least 18 GiB of free runtime storage is needed for the selected GGUF components.')
print('T4 preflight passed.')


GPU: Tesla T4 | total: 15360 MiB | used: 0 MiB
Free /content storage: 65.2 GiB
T4 preflight passed.


## Prepare the private local runtime

This downloads audio.cpp's official CUDA server prebuilt for compute capability 7.5 (T4), rather than compiling it. It does not install PyTorch, TorchAudio, Qwen-TTS, Gradio, or the packages that caused the previous ABI error. Tencent's repository is cloned only to retrieve the official short demo WAVs.

In [ ]:
# Fast T4 setup: use audio.cpp's official SM75 Colab prebuilt server instead of compiling it.
%cd /content
!apt-get -qq update
!apt-get -qq install -y ca-certificates curl ffmpeg libgomp1 libssl3
!if [ ! -d /content/AuK/.git ]; then git clone --depth 1 https://github.com/Tencent-Hunyuan/AuK.git /content/AuK; fi

import hashlib
import json
import shutil
import tarfile
import urllib.request
from pathlib import Path

RELEASE_API = 'https://api.github.com/repos/0xShug0/audio.cpp/releases/latest'
PREBUILT_SUFFIX = '-bin-ubuntu-x64-cuda12.8-colab.tar.gz'
PREBUILT_ROOT = Path('/content/audio-cpp-prebuilt')
request = urllib.request.Request(
    RELEASE_API,
    headers={'Accept': 'application/vnd.github+json', 'User-Agent': 'auk-t4-colab'},
)
with urllib.request.urlopen(request, timeout=30) as response:
    release = json.load(response)
matches = [asset for asset in release.get('assets', []) if asset['name'].endswith(PREBUILT_SUFFIX)]
if len(matches) != 1:
    raise RuntimeError('audio.cpp did not publish exactly one current CUDA 12.8 Colab prebuilt. Do not wait for a source build; retry later.')
asset = matches[0]
expected_sha256 = asset.get('digest', '').removeprefix('sha256:')
if len(expected_sha256) != 64:
    raise RuntimeError('The official prebuilt did not provide a SHA-256 digest.')
archive_path = Path('/content') / asset['name']
print('Downloading official T4/SM75 CUDA prebuilt:', asset['name'])
urllib.request.urlretrieve(asset['browser_download_url'], archive_path)
digest = hashlib.sha256()
with archive_path.open('rb') as stream:
    for block in iter(lambda: stream.read(1024 * 1024), b''):
        digest.update(block)
if digest.hexdigest() != expected_sha256:
    raise RuntimeError('The downloaded audio.cpp prebuilt failed SHA-256 verification.')
if PREBUILT_ROOT.exists():
    shutil.rmtree(PREBUILT_ROOT)
PREBUILT_ROOT.mkdir(parents=True)
with tarfile.open(archive_path, 'r:gz') as archive:
    root = PREBUILT_ROOT.resolve()
    for member in archive.getmembers():
        member_path = (PREBUILT_ROOT / member.name).resolve()
        if root not in member_path.parents and member_path != root:
            raise RuntimeError('The prebuilt archive contains an unsafe path.')
    archive.extractall(PREBUILT_ROOT, filter='data')
archive_path.unlink(missing_ok=True)
server_matches = [path for path in PREBUILT_ROOT.rglob('audiocpp_server') if path.is_file()]
if len(server_matches) != 1:
    raise RuntimeError(f'The official prebuilt should contain one audiocpp_server executable file; found: {server_matches}')
AUDIOCPP_SERVER = server_matches[0]
AUDIOCPP_SERVER.chmod(0o755)
print('Ready:', AUDIOCPP_SERVER)
subprocess.run([str(AUDIOCPP_SERVER), '--help'], check=True)


/content
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../libssl-dev_3.0.13-0ubuntu3.16_amd64.deb ...
Unpacking libssl-dev:amd64 (3.0.13-0ubuntu3.16) over (3.0.13-0ubuntu3.15) ...
Preparing to unpack .../libssl3t64_3.0.13-0ubuntu3.16_amd64.deb ...
Unpacking libssl3t64:amd64 (3.0.13-0ubuntu3.16) over (3.0.13-0ubuntu3.15) ...
Setting up libssl3t64:amd64 (3.0.13-0ubuntu3.16) ...
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../openssl_3.0.13-0ubuntu3.16_amd64.deb ...
Unpacking openssl (3.0.13-0ubuntu3

CompletedProcess(args=['/content/audio-cpp-prebuilt/audiocpp_server', '--help'], returncode=0)

In [ ]:
# The official T4 prebuilt intentionally ships audiocpp_server (not the CLI).
server_candidates = [path for path in PREBUILT_ROOT.rglob('audiocpp_server') if path.is_file()]
if len(server_candidates) != 1:
    raise RuntimeError(f'Expected one server executable; found: {server_candidates}')
AUDIOCPP_SERVER = server_candidates[0]
AUDIOCPP_SERVER.chmod(0o755)
subprocess.run([str(AUDIOCPP_SERVER), '--help'], check=True)
print('Using local server:', AUDIOCPP_SERVER)

# The existing profile check expects this path; the server runner below supersedes the old CLI helper.
compat_cli = Path('/content/audio.cpp/build/linux-cuda-release/bin/audiocpp_cli')
compat_cli.parent.mkdir(parents=True, exist_ok=True)
if compat_cli.exists() or compat_cli.is_symlink():
    compat_cli.unlink()
compat_cli.symlink_to(AUDIOCPP_SERVER)


Using local server: /content/audio-cpp-prebuilt/audiocpp_server


## Download the T4 model components

The default is AuK Base F32 + Qwen Q8_0 + AuK VAE F32. A Base-Q8 file is also downloaded as an explicit lower-memory retry option. These public files stay only in the Colab runtime. An unauthenticated Hugging Face warning is normal; a free personal token can make downloads more reliable, but never paste a token into a shared notebook.

In [ ]:
%pip -q install -U huggingface_hub hf_xet

from pathlib import Path
from huggingface_hub import snapshot_download

MODEL_DIR = Path('/content/models/AuK-Base-and-Flash-GGUF')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
snapshot_download(
    repo_id='audio-cpp/AuK-Base-and-Flash-GGUF',
    local_dir=str(MODEL_DIR),
    allow_patterns=[
        'config/*', 'tokenizer/*', 'auk-base-f32.gguf', 'auk-base-q8_0.gguf',
        'qwen2.5-omni-3b-q8_0.gguf', 'auk-vae-f32.gguf',
    ],
)
expected = [
    'config/auk-base.yaml', 'tokenizer/tokenizer.json', 'auk-base-f32.gguf',
    'auk-base-q8_0.gguf', 'qwen2.5-omni-3b-q8_0.gguf', 'auk-vae-f32.gguf',
]
missing = [name for name in expected if not (MODEL_DIR / name).is_file()]
if missing:
    raise RuntimeError(f'Model download is incomplete: {missing}')
for name in expected[2:]:
    print(f'{name}: {(MODEL_DIR / name).stat().st_size / 2**30:.2f} GiB')
print('All selected T4 components are present.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.5/828.5 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.0.0 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.0.0 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.0.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.0.0 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.0.0 which is incompatible.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

auk-base-f32.gguf: 5.70 GiB
auk-base-q8_0.gguf: 1.53 GiB
qwen2.5-omni-3b-q8_0.gguf: 3.96 GiB
auk-vae-f32.gguf: 0.59 GiB
All selected T4 components are present.


## Configure the T4 profile

Leave quality_first_f32_base selected. It retains the F32 AuK Base generator and VAE; only Qwen is quantized to give a T4 a chance to run. If it OOMs, the runner visibly reports the change and tries Base-Q8 once. The actual profile is printed beside every audio pair and stored in the manifest.

In [ ]:
from pathlib import Path

CLI = Path('/content/audio.cpp/build/linux-cuda-release/bin/audiocpp_cli')
AUK_REPO = Path('/content/AuK')
ASSET_DIR = AUK_REPO / 'assets/demo-input-audio'
RESULT_DIR = Path('/content/auk_t4_results')
RESULT_DIR.mkdir(exist_ok=True)

t4_profile = 'quality_first_f32_base'  # @param ['quality_first_f32_base', 'compact_q8_base']
retry_with_compact_on_oom = True  # @param {type:'boolean'}

PROFILES = {
    'quality_first_f32_base': {
        'generator': 'auk-base-f32.gguf',
        'qwen': 'qwen2.5-omni-3b-q8_0.gguf',
        'vae': 'auk-vae-f32.gguf',
        'label': 'quality-first: Base F32 + Qwen Q8_0 + VAE F32',
    },
    'compact_q8_base': {
        'generator': 'auk-base-q8_0.gguf',
        'qwen': 'qwen2.5-omni-3b-q8_0.gguf',
        'vae': 'auk-vae-f32.gguf',
        'label': 'lower-memory experimental: Base Q8_0 + Qwen Q8_0 + VAE F32',
    },
}
if t4_profile not in PROFILES:
    raise ValueError(f'Unknown profile: {t4_profile}')
if not CLI.is_file():
    raise RuntimeError('The audio.cpp CLI is missing. Re-run the build cell.')
if not ASSET_DIR.is_dir():
    raise RuntimeError('The Tencent AuK demo assets are missing. Re-run the build cell.')
for component in PROFILES[t4_profile].values():
    if isinstance(component, str) and component.endswith('.gguf') and not (MODEL_DIR / component).is_file():
        raise RuntimeError(f'Missing model component: {component}')
print('Selected:', PROFILES[t4_profile]['label'])
print('This is local T4 inference. No public Gradio Space is used.')


Selected: quality-first: Base F32 + Qwen Q8_0 + VAE F32
This is local T4 inference. No public Gradio Space is used.


## Start the private local AuK server

The T4 prebuilt distributes the local audio.cpp server rather than the command-line client. This cell starts it only on 127.0.0.1, configures the selected AuK components, and keeps the audio in the Colab runtime. It does not create a public tunnel.

In [ ]:
import time
import urllib.request

SERVER_PORT = 8765
SERVER_URL = f'http://127.0.0.1:{SERVER_PORT}'
SERVER_CONFIG_PATH = Path('/content/auk_t4_server.json')
SERVER_LOG_PATH = Path('/content/auk_t4_server.log')


def start_auk_server(profile_name):
    global AUK_SERVER_PROCESS, AUK_SERVER_PROFILE
    previous = globals().get('AUK_SERVER_PROCESS')
    if previous is not None and previous.poll() is None:
        previous.terminate()
        try:
            previous.wait(timeout=15)
        except subprocess.TimeoutExpired:
            previous.kill()
            previous.wait()
    profile = PROFILES[profile_name]
    config = {
        'host': '127.0.0.1',
        'port': SERVER_PORT,
        'backend': 'cuda',
        'device': 0,
        'threads': 1,
        'lazy_load': True,
        'max_loaded_models': 1,
        'model_spec_override': str(PREBUILT_ROOT / 'model_specs' / 'auk.json'),
        'models': [{
            'id': 'auk-t4',
            'family': 'auk',
            'path': str(MODEL_DIR),
            'task': 'gen',
            'mode': 'offline',
            'session_options': {
                'auk.variant': 'base',
                'auk.model_gguf': profile['generator'],
                'auk.qwen_gguf': profile['qwen'],
                'auk.vae_gguf': profile['vae'],
                'auk.mem_saver': 'true',
            },
            'default_request_options': {
                'num_inference_steps': 32,
                'guidance_scale': 2.0,
                'sway_sampling_coef': -1.0,
            },
        }],
    }
    SERVER_CONFIG_PATH.write_text(json.dumps(config, indent=2))
    SERVER_LOG_PATH.write_text('')
    AUK_SERVER_PROCESS = subprocess.Popen(
        [str(AUDIOCPP_SERVER), '--config', str(SERVER_CONFIG_PATH), '--no-ui'],
        cwd=str(AUDIOCPP_SERVER.parent),
        stdout=SERVER_LOG_PATH.open('w'),
        stderr=subprocess.STDOUT,
        text=True,
    )
    deadline = time.monotonic() + 90
    while time.monotonic() < deadline:
        if AUK_SERVER_PROCESS.poll() is not None:
            raise RuntimeError('The local server stopped during startup:\n' + SERVER_LOG_PATH.read_text(errors='replace')[-4000:])
        try:
            with urllib.request.urlopen(f'{SERVER_URL}/health', timeout=2) as response:
                if response.status == 200:
                    AUK_SERVER_PROFILE = profile_name
                    print('Local AuK server is ready with:', PROFILES[profile_name]['label'])
                    return
        except Exception:
            time.sleep(1)
    raise RuntimeError('The local AuK server did not become ready:\n' + SERVER_LOG_PATH.read_text(errors='replace')[-4000:])

start_auk_server(t4_profile)


Local AuK server is ready with: quality-first: Base F32 + Qwen Q8_0 + VAE F32


In [ ]:
import hashlib
import json
import subprocess
import time
from html import escape
from shutil import copy2
from IPython.display import Audio, HTML, display

MANIFEST_PATH = RESULT_DIR / 'manifest.json'
RUNS = {}

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def inspect_audio(path):
    raw = subprocess.check_output([
        'ffprobe', '-v', 'error', '-show_entries', 'format=duration',
        '-show_entries', 'stream=codec_type,sample_rate,channels', '-of', 'json', str(path),
    ], text=True)
    info = json.loads(raw)
    stream = next((item for item in info.get('streams', []) if item.get('codec_type') == 'audio'), None)
    if stream is None:
        raise RuntimeError(f'No audio stream found in {path.name}')
    return {
        'sample_rate': int(stream.get('sample_rate') or 0),
        'channels': int(stream.get('channels') or 0),
        'duration_seconds': float(info.get('format', {}).get('duration') or 0),
        'bytes': path.stat().st_size,
    }

def validate_auk_output(path):
    if not path.is_file() or path.stat().st_size < 1024:
        raise RuntimeError(f'AuK did not create a usable WAV: {path}')
    meta = inspect_audio(path)
    if meta['sample_rate'] != 24000 or meta['channels'] != 1:
        raise RuntimeError(f'Expected mono 24 kHz output, got {meta}')
    if not 0.1 < meta['duration_seconds'] < 90:
        raise RuntimeError(f'Output duration is not plausible: {meta}')
    return meta

def build_auk_command(profile_name, instruction, input_path, output_path, duration_seconds, seed):
    profile = PROFILES[profile_name]
    command = [
        str(CLI), '--task', 'gen', '--family', 'auk', '--model', str(MODEL_DIR), '--backend', 'cuda',
        '--audio', str(input_path), '--text', instruction, '--seed', str(seed), '--out', str(output_path), '--log',
        '--session-option', 'auk.variant=base',
        '--session-option', f"auk.model_gguf={profile['generator']}",
        '--session-option', f"auk.qwen_gguf={profile['qwen']}",
        '--session-option', f"auk.vae_gguf={profile['vae']}",
        '--session-option', 'auk.mem_saver=true',
        '--request-option', 'num_inference_steps=32',
        '--request-option', 'guidance_scale=2.0',
        '--request-option', 'sway_sampling_coef=-1.0',
    ]
    if duration_seconds is not None:
        command.extend(['--request-option', f'duration_sec={float(duration_seconds):.2f}'])
    return command

def looks_like_oom(log_text):
    text = log_text.lower()
    return any(flag in text for flag in ('out of memory', 'cuda_error_memory_allocation', 'cuda error: out of memory', 'graph allocation failed'))

def write_manifest():
    MANIFEST_PATH.write_text(json.dumps({'runs': RUNS}, indent=2, ensure_ascii=False))

def run_auk_t4(result_id, instruction, input_path, output_name, *, transcript, change, duration_seconds=None, seed=42):
    input_path = Path(input_path)
    output_path = RESULT_DIR / Path(output_name).name
    if not input_path.is_file():
        raise FileNotFoundError(input_path)
    attempts = [t4_profile]
    if retry_with_compact_on_oom and t4_profile == 'quality_first_f32_base':
        attempts.append('compact_q8_base')
    started = time.time()
    last_log = ''
    used_profile = None
    for attempt, profile_name in enumerate(attempts, 1):
        print(f'Attempt {attempt}/{len(attempts)} — {PROFILES[profile_name]["label"]}')
        print('Instruction:', instruction)
        completed = subprocess.run(
            build_auk_command(profile_name, instruction, input_path, output_path, duration_seconds, seed),
            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        )
        last_log = completed.stdout or ''
        print(last_log[-8000:] if last_log else '(The CLI returned no log text.)')
        if completed.returncode == 0:
            used_profile = profile_name
            break
        if attempt == 1 and profile_name == 'quality_first_f32_base' and looks_like_oom(last_log):
            print('IMPORTANT: F32 Base exceeded T4 memory. Retrying once with the clearly labelled lower-memory Base-Q8 profile.')
            continue
        raise RuntimeError(f'audio.cpp exited with code {completed.returncode}. Read the CLI log printed above.')
    if used_profile is None:
        raise RuntimeError(f'AuK did not complete. Last log: {last_log[-2500:]}')
    output_meta = validate_auk_output(output_path)
    input_meta = inspect_audio(input_path)
    log_path = RESULT_DIR / f'{result_id}-cli.log.txt'
    log_path.write_text(last_log)
    result = {
        'id': result_id,
        'input_file': input_path.name,
        'output_file': output_path.name,
        'instruction': instruction,
        'transcript': transcript,
        'intended_change': change,
        'seed': seed,
        'requested_duration_seconds': duration_seconds,
        'profile_used': used_profile,
        'profile_description': PROFILES[used_profile]['label'],
        'input_audio': input_meta,
        'output_audio': output_meta,
        'input_sha256': sha256_file(input_path),
        'output_sha256': sha256_file(output_path),
        'elapsed_seconds': round(time.time() - started, 1),
        'human_review': 'pending — listen for naturalness, identity, instruction adherence, and artifacts',
        'log_file': log_path.name,
    }
    RUNS[result_id] = result
    write_manifest()
    print(f'Saved {output_path.name} in {result["elapsed_seconds"]:.1f}s | {output_meta["duration_seconds"]:.2f}s, mono 24 kHz')
    print('Profile actually used:', result['profile_description'])
    return output_path, result

def highlight(text, phrase):
    return escape(text).replace(escape(phrase), f'<mark>{escape(phrase)}</mark>', 1)

def show_pair(title, input_path, output_path, details_html, result):
    display(HTML(
        f'<h3>{escape(title)}</h3>{details_html}'
        f'<p><small><b>Actual model profile:</b> {escape(result["profile_description"])} | '
        f'<b>seed:</b> {result["seed"]} | <b>output:</b> {result["output_audio"]["duration_seconds"]:.2f}s / mono 24 kHz</small></p>'
    ))
    print('INPUT')
    display(Audio(filename=str(input_path)))
    print('TENCENT AUK OUTPUT — listen before approving')
    display(Audio(filename=str(output_path)))


In [ ]:
# Override the earlier CLI helper: the official T4 prebuilt uses the local server API.
import base64
from urllib.error import HTTPError, URLError


def post_server_json(path, body):
    request = urllib.request.Request(
        f'{SERVER_URL}{path}',
        data=json.dumps(body).encode('utf-8'),
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    with urllib.request.urlopen(request, timeout=1800) as response:
        return json.loads(response.read().decode('utf-8'))


def run_auk_t4(result_id, instruction, input_path, output_name, *, transcript, change, duration_seconds=None, seed=42):
    input_path = Path(input_path)
    output_path = RESULT_DIR / Path(output_name).name
    if not input_path.is_file():
        raise FileNotFoundError(input_path)
    attempts = [t4_profile]
    if retry_with_compact_on_oom and t4_profile == 'quality_first_f32_base':
        attempts.append('compact_q8_base')
    started = time.time()
    response_json = None
    used_profile = None
    for attempt, profile_name in enumerate(attempts, 1):
        if globals().get('AUK_SERVER_PROFILE') != profile_name:
            start_auk_server(profile_name)
        request_options = {}
        if duration_seconds is not None:
            request_options['duration_sec'] = float(duration_seconds)
        body = {
            'model': 'auk-t4',
            'request': {
                'audio': str(input_path),
                'text': instruction,
                'seed': seed,
                'options': request_options,
            },
        }
        print(f'Attempt {attempt}/{len(attempts)} — {PROFILES[profile_name]["label"]}')
        print('Instruction:', instruction)
        try:
            response_json = post_server_json('/v1/tasks/run', body)
            used_profile = profile_name
            break
        except HTTPError as exc:
            error_text = exc.read().decode('utf-8', errors='replace')
            print(f'Server returned HTTP {exc.code}: {error_text}')
            if attempt == 1 and profile_name == 'quality_first_f32_base' and looks_like_oom(error_text):
                print('IMPORTANT: F32 Base exceeded T4 memory. Retrying once with the clearly labelled Base-Q8 profile.')
                continue
            raise RuntimeError(f'Local AuK server failed: HTTP {exc.code}: {error_text}') from exc
        except URLError as exc:
            log_tail = SERVER_LOG_PATH.read_text(errors='replace')[-4000:]
            raise RuntimeError(f'Could not reach the local AuK server: {exc}\n{log_tail}') from exc
    if response_json is None or used_profile is None:
        raise RuntimeError('AuK did not return a result.')
    if 'audio' not in response_json:
        raise RuntimeError(f'AuK server returned no audio: {response_json}')
    output_path.write_bytes(base64.b64decode(response_json['audio']))
    output_meta = validate_auk_output(output_path)
    input_meta = inspect_audio(input_path)
    log_path = RESULT_DIR / f'{result_id}-server-response.json'
    log_path.write_text(json.dumps(response_json, indent=2))
    result = {
        'id': result_id,
        'input_file': input_path.name,
        'output_file': output_path.name,
        'instruction': instruction,
        'transcript': transcript,
        'intended_change': change,
        'seed': seed,
        'requested_duration_seconds': duration_seconds,
        'profile_used': used_profile,
        'profile_description': PROFILES[used_profile]['label'],
        'input_audio': input_meta,
        'output_audio': output_meta,
        'input_sha256': sha256_file(input_path),
        'output_sha256': sha256_file(output_path),
        'elapsed_seconds': round(time.time() - started, 1),
        'human_review': 'pending — listen for naturalness, identity, instruction adherence, and artifacts',
        'log_file': log_path.name,
    }
    RUNS[result_id] = result
    write_manifest()
    print(f'Saved {output_path.name} in {result["elapsed_seconds"]:.1f}s | {output_meta["duration_seconds"]:.2f}s, mono 24 kHz')
    print('Profile actually used:', result['profile_description'])
    return output_path, result


## Three presentation examples

Examples 1–3 intentionally share one verified English source sentence. This controls the comparison: the listener can judge whether the same voice and words survive three different transformations, and the notebook can show an exact transcript rather than an unverified automatic transcription. Run the cells one at a time; each output is generated locally on the T4.

In [ ]:
# Example 1 — transcript / content edit
content_source = ASSET_DIR / 'content-edit/content.wav'
original_transcript = 'Do you think the biggest victory is not fighting for what we want, but accepting what we cannot have?'
old_words = 'but accepting what we cannot have'
new_words = 'and living well with dreams unmet'
edited_transcript = original_transcript.replace(old_words, new_words)
instruction = f"Replace '{old_words}' with '{new_words}'."

transcript_input = RESULT_DIR / '01-transcript-input.wav'
copy2(content_source, transcript_input)
transcript_output, transcript_result = run_auk_t4(
    '01-transcript', instruction, transcript_input, '01-transcript-output.wav',
    transcript=original_transcript,
    change=f'{old_words} → {new_words}',
    duration_seconds=7.0,
)
show_pair(
    'Example 1 — transcript edit', transcript_input, transcript_output,
    '<p><b>Original transcript:</b> ' + highlight(original_transcript, old_words) + '</p>'
    + f'<p><b>Changed:</b> <mark>{escape(old_words)}</mark> → <mark>{escape(new_words)}</mark></p>'
    + '<p><b>Edited transcript:</b> ' + highlight(edited_transcript, new_words) + '</p>'
    + f'<p><b>Exact AuK instruction:</b> <code>{escape(instruction)}</code></p>',
    transcript_result,
)


Attempt 1/2 — quality-first: Base F32 + Qwen Q8_0 + VAE F32
Instruction: Replace 'but accepting what we cannot have' with 'and living well with dreams unmet'.
Saved 01-transcript-output.wav in 82.0s | 7.00s, mono 24 kHz
Profile actually used: quality-first: Base F32 + Qwen Q8_0 + VAE F32


INPUT


TENCENT AUK OUTPUT — listen before approving


In [ ]:
# Example 2 — tone / emotion edit
tone_input = RESULT_DIR / '02-tone-input.wav'
copy2(content_source, tone_input)
tone_instruction = 'Change the emotion to excited.'
tone_output, tone_result = run_auk_t4(
    '02-tone', tone_instruction, tone_input, '02-tone-output.wav',
    transcript=original_transcript,
    change='delivery: original → excited; the words are intended to remain unchanged',
)
show_pair(
    'Example 2 — tone / emotion edit', tone_input, tone_output,
    f'<p><b>Original transcript:</b> {escape(original_transcript)}</p>'
    + '<p><b>Changed:</b> <mark>delivery: original → excited</mark>. The words are intended to stay the same.</p>'
    + f'<p><b>Exact AuK instruction:</b> <code>{escape(tone_instruction)}</code></p>',
    tone_result,
)


Attempt 1/2 — quality-first: Base F32 + Qwen Q8_0 + VAE F32
Instruction: Change the emotion to excited.
Saved 02-tone-output.wav in 87.9s | 7.00s, mono 24 kHz
Profile actually used: quality-first: Base F32 + Qwen Q8_0 + VAE F32


INPUT


TENCENT AUK OUTPUT — listen before approving


In [ ]:
# Example 3 — controlled speech cleanup
# Input is intentionally made noisy/reverberant from the verified English source.
cleanup_input = RESULT_DIR / '03-cleanup-input.wav'
clean_meta = inspect_audio(content_source)
duration = clean_meta['duration_seconds']
filter_graph = (
    f'[0:a]aecho=0.80:0.55:65:0.30[reverb];'
    f'anoisesrc=color=pink:amplitude=0.025:r=24000,atrim=duration={duration:.3f}[noise];'
    f'[reverb][noise]amix=inputs=2:weights=1 0.65:normalize=0,'
    f'alimiter=limit=0.95,atrim=duration={duration:.3f}[mixed]'
)
subprocess.run([
    'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y', '-i', str(content_source),
    '-filter_complex', filter_graph, '-map', '[mixed]', '-ac', '1', '-ar', '24000',
    '-c:a', 'pcm_s16le', str(cleanup_input),
], check=True)
cleanup_instruction = 'Preserve all speakers, remove noise and reverberation, and output clean speech of the same length.'
cleanup_output, cleanup_result = run_auk_t4(
    '03-cleanup', cleanup_instruction, cleanup_input, '03-cleanup-output.wav',
    transcript=original_transcript,
    change='synthetic background noise + room reverb → reduced; no words intentionally changed',
)
show_pair(
    'Example 3 — speech cleanup', cleanup_input, cleanup_output,
    f'<p><b>Original transcript:</b> {escape(original_transcript)}</p>'
    + '<p><b>Changed:</b> <mark>synthetic background noise + room reverb → reduced</mark>. No words are intentionally changed.</p>'
    + '<p><i>The input was deliberately degraded from Tencent\'s clean English sample; it is not presented as a real-world recording.</i></p>'
    + f'<p><b>Exact AuK instruction:</b> <code>{escape(cleanup_instruction)}</code></p>',
    cleanup_result,
)


Attempt 1/2 — quality-first: Base F32 + Qwen Q8_0 + VAE F32
Instruction: Preserve all speakers, remove noise and reverberation, and output clean speech of the same length.
Saved 03-cleanup-output.wav in 94.4s | 7.00s, mono 24 kHz
Profile actually used: quality-first: Base F32 + Qwen Q8_0 + VAE F32


INPUT


TENCENT AUK OUTPUT — listen before approving


## Fourth capability — speaker isolation

This uses Tencent's official multi-speaker sample and asks AuK to keep the person who begins speaking second. Choosing by speaking order is more reliable than identifying a target by a phrase.

In [ ]:
# Example 4 — speaker isolation
isolation_source = ASSET_DIR / 'ss/zh-1-input.wav'
isolation_input = RESULT_DIR / '04-isolation-input.wav'
copy2(isolation_source, isolation_input)
isolation_instruction = (
    'Please keep the second speaker to start talking and remove the other speakers, '
    'outputting a single clean speech track.'
)
isolation_output, isolation_result = run_auk_t4(
    '04-isolation', isolation_instruction, isolation_input, '04-isolation-output.wav',
    transcript='Official multi-speaker Mandarin sample; no verbatim transcript is claimed here.',
    change='all other speakers → removed; keep the second speaker to begin talking',
)
show_pair(
    'Example 4 — speaker isolation', isolation_input, isolation_output,
    '<p><b>Target:</b> <mark>the second speaker to start talking</mark>.</p>'
    + '<p><b>Changed:</b> other speakers → <mark>removed</mark>.</p>'
    + f'<p><b>Exact AuK instruction:</b> <code>{escape(isolation_instruction)}</code></p>',
    isolation_result,
)


Attempt 1/2 — quality-first: Base F32 + Qwen Q8_0 + VAE F32
Instruction: Please keep the second speaker to start talking and remove the other speakers, outputting a single clean speech track.
Saved 04-isolation-output.wav in 209.7s | 19.22s, mono 24 kHz
Profile actually used: quality-first: Base F32 + Qwen Q8_0 + VAE F32


INPUT


TENCENT AUK OUTPUT — listen before approving


In [ ]:
# @title Mark the results you personally listened to and approve
approved_ids_text = '01-transcript, 02-tone, 03-cleanup, 04-isolation'  # @param {type:'string'}
approved_ids = {item.strip() for item in approved_ids_text.split(',') if item.strip()}
if not RUNS:
    raise RuntimeError('No results exist yet. Run one or more example cells first.')
unknown = approved_ids.difference(RUNS)
if unknown:
    raise ValueError(f'Unknown IDs: {sorted(unknown)}. Available: {sorted(RUNS)}')
for result_id, result in RUNS.items():
    result['human_review'] = 'approved by notebook user after listening' if result_id in approved_ids else 'not approved / do not share as final'
write_manifest()
print('Approved:', sorted(approved_ids) or '(none)')
print('Available:', sorted(RUNS))


Approved: ['01-transcript', '02-tone', '03-cleanup', '04-isolation']
Available: ['01-transcript', '02-tone', '03-cleanup', '04-isolation']


In [ ]:
# Download only pairs you approved in the review cell.
import zipfile
from google.colab import files

approved = [item for item in RUNS.values() if item['human_review'].startswith('approved')]
if not approved:
    raise RuntimeError('No pairs are approved. Listen first, enter the IDs in the review cell, and run it again.')
archive_path = RESULT_DIR / 'tencent_auk_t4_approved_results.zip'
readme_lines = [
    'Tencent AuK T4 experimental local GGUF results', '',
    'These pairs were explicitly approved by the notebook user after listening.',
    'Runtime: audio.cpp experimental AuK family; output is mono 24 kHz.',
    'The manifest records instructions, model profile, seed, hashes, durations, and review status.', '',
]
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for result in approved:
        for filename in (result['input_file'], result['output_file'], result['log_file']):
            path = RESULT_DIR / filename
            if path.is_file():
                archive.write(path, arcname=path.name)
        readme_lines.extend([
            f"{result['id']}: {result['input_file']} → {result['output_file']}",
            f"  Change: {result['intended_change']}",
            f"  Profile: {result['profile_description']} | seed: {result['seed']}", '',
        ])
    archive.write(MANIFEST_PATH, arcname='manifest.json')
    archive.writestr('README.txt', '\n'.join(readme_lines))
print('Created:', archive_path)
files.download(str(archive_path))


Created: /content/auk_t4_results/tencent_auk_t4_approved_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Troubleshooting and honest expectations

- No T4 or no CUDA: reconnect after selecting T4 GPU. The notebook stops instead of silently running on CPU.
- Build fails: start a fresh T4 runtime and rerun the build cell. This route avoids the incompatible Python/TorchAudio environment in the old notebook.
- Model download fails: rerun the download cell. The model is public; an optional personal Hugging Face token can improve reliability.
- CUDA out of memory: keep clips below roughly 8 seconds and run one request at a time. The runner visibly retries Base-Q8 only after a Base-F32 OOM; you may explicitly select compact_q8_base for later requests.
- Unnatural output, wrong voice, missed emotion, remaining noise, or wrong speaker: do not mark that result approved. A successful command does not certify production quality.

References: Tencent AuK cookbook: https://github.com/Tencent-Hunyuan/AuK/blob/main/docs/COOKBOOK.md . audio.cpp AuK guide: https://github.com/0xShug0/audio.cpp/blob/main/docs/community_models/auk.md . AuK GGUF model card: https://huggingface.co/audio-cpp/AuK-Base-and-Flash-GGUF . T4-capable audio.cpp Colab: https://colab.research.google.com/github/0xShug0/audio.cpp/blob/main/Notebooks/colab_audio_cpp.ipynb .